# Deep Q-learning avec cibles-Q fixes — CartPole-v1 à partir des images

Ce carnet reprend le script du chapitre *Apprentissage par renforcement* du
manuel. L'état de l'environnement est la **différence entre les deux dernières
images** produites par le simulateur, à la manière des travaux de DeepMind.

## Dépendances

L'environnement doit **produire des images** (`render_mode="rgb_array"`). Ce
rendu est assuré par `pygame`, qui n'est pas installé par défaut avec
Gymnasium. Exécutez la cellule suivante une seule fois, puis redémarrez le
noyau si `pygame` vient d'être installé.


In [ ]:
# Dépendances (à n'exécuter qu'une fois).
# Les crochets sont essentiels : ils installent pygame, requis par le rendu
# graphique de CartPole (render_mode="rgb_array").
%pip install "gymnasium[classic-control]" torch matplotlib

# Vérification : ces deux imports doivent réussir avant d'aller plus loin.
import gymnasium as gym
import pygame

print("gymnasium", gym.__version__, "| pygame", pygame.version.ver)
gym.make("CartPole-v1", render_mode="rgb_array").close()
print("Le rendu graphique est disponible.")


In [ ]:
"""
Deep Q-learning avec cibles-Q fixes
CartPole-v1, états = différence d'images d'écran (sans torchvision)
"""

import gymnasium as gym
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import namedtuple
from itertools import count
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

env = gym.make("CartPole-v1", render_mode="rgb_array")

Observation = namedtuple(
    "Observation", ("etat", "action", "etat_suivant", "recompense")
)


class HistoriqueObservations(object):
    """Mémoire circulaire des observations pour l'apprentissage."""

    def __init__(self, taille_memoire):
        self.taille_memoire = taille_memoire
        self.indice_courant = 0
        self.liste_observations = []

    def ajouter_historique(self, *args):
        if len(self.liste_observations) < self.taille_memoire:
            self.liste_observations.append(None)
        self.liste_observations[self.indice_courant] = Observation(*args)
        self.indice_courant = (self.indice_courant + 1) % self.taille_memoire

    def mini_lot_observations(self, taille_mini_lot):
        return random.sample(self.liste_observations, taille_mini_lot)

    def __len__(self):
        return len(self.liste_observations)


class RNAQ(nn.Module):
    """
    Prend un état image (différence de deux écrans, 3 canaux) et produit Q
    pour chaque action.
    """

    def __init__(self, hauteur, largeur, nb_actions_y):
        super(RNAQ, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, stride=2)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, stride=2)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=5, stride=2)
        self.bn3 = nn.BatchNorm2d(32)

        def taille_sortie_convolution(taille, kernel_size=5, stride=2):
            return (taille - (kernel_size - 1) - 1) // stride + 1

        largeur_conv = taille_sortie_convolution(
            taille_sortie_convolution(taille_sortie_convolution(largeur))
        )
        hauteur_conv = taille_sortie_convolution(
            taille_sortie_convolution(taille_sortie_convolution(hauteur))
        )
        taille_X_lineaire = largeur_conv * hauteur_conv * 32
        self.couche_finale = nn.Linear(taille_X_lineaire, nb_actions_y)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return self.couche_finale(x.view(x.size(0), -1))


def calculer_position_chariot(largeur_ecran):
    largeur_monde = env.unwrapped.x_threshold * 2
    echelle = largeur_ecran / largeur_monde
    return int(env.unwrapped.state[0] * echelle + largeur_ecran / 2.0)


def redimensionner_ecran(ecran, taille=40):
    """
    Redimensionne un tenseur (C, H, W) vers (C, taille, taille)
    sans torchvision.
    """
    ecran = ecran.unsqueeze(0)
    ecran = F.interpolate(
        ecran, size=(taille, taille), mode="bilinear", align_corners=False
    )
    return ecran.squeeze(0)


def chercher_ecran():
    """Image d'écran recadrée autour du chariot, forme (1, C, H, W)."""
    ecran = env.render().transpose((2, 0, 1))
    _, hauteur_ecran, largeur_ecran = ecran.shape
    ecran = ecran[:, int(hauteur_ecran * 0.4) : int(hauteur_ecran * 0.8)]
    largeur_vue = int(largeur_ecran * 0.6)
    position_chariot = calculer_position_chariot(largeur_ecran)
    if position_chariot < largeur_vue // 2:
        portee_tranche = slice(largeur_vue)
    elif position_chariot > (largeur_ecran - largeur_vue // 2):
        portee_tranche = slice(-largeur_vue, None)
    else:
        portee_tranche = slice(
            position_chariot - largeur_vue // 2,
            position_chariot + largeur_vue // 2,
        )
    ecran = ecran[:, :, portee_tranche]
    ecran = np.ascontiguousarray(ecran, dtype=np.float32) / 255.0
    ecran = torch.from_numpy(ecran)
    return redimensionner_ecran(ecran).unsqueeze(0)


env.reset(seed=42)
plt.figure()
plt.imshow(
    chercher_ecran().cpu().squeeze(0).permute(1, 2, 0).numpy(),
    interpolation="none",
)
plt.title("Exemple d'écran après transformations")
plt.show()

ecran_initial = chercher_ecran()
_, _, hauteur_ecran, largeur_ecran = ecran_initial.shape

n_actions = env.action_space.n
rnaq = RNAQ(hauteur_ecran, largeur_ecran, n_actions)
rnaq_cible = RNAQ(hauteur_ecran, largeur_ecran, n_actions)
rnaq_cible.load_state_dict(rnaq.state_dict())
rnaq_cible.eval()
optimiseur = optim.RMSprop(rnaq.parameters())
liste_observations = HistoriqueObservations(3000)


def choisir_action(etat, epsilon):
    """Politique epsilon-vorace dérivée de rnaq."""
    if random.uniform(0, 1) > epsilon:
        with torch.no_grad():
            return rnaq(etat).max(1)[1].view(1, 1)
    return torch.tensor([[random.randrange(n_actions)]], dtype=torch.long)


def afficher_longueur_episode(longueur_episode, fenetre=10):
    plt.figure(figsize=(12, 6))
    plt.plot(longueur_episode, label="Longueur épisode")
    if longueur_episode.shape[0] > fenetre:
        longueur_moyenne_fenetre = [
            longueur_episode[i : i + fenetre].mean()
            for i in range(longueur_episode.shape[0] - fenetre)
        ]
        plt.plot(
            np.arange(fenetre, longueur_episode.shape[0]),
            longueur_moyenne_fenetre,
            label="Moyenne mobile",
        )
    plt.xlabel("Épisode")
    plt.ylabel("Longueur épisode")
    plt.title(
        "DQN (images), CartPole : longueur d'épisode, fenêtre="
        + str(fenetre)
    )
    plt.legend(loc="upper left")
    plt.show()


def optimisation_RNAQ(taille_mini_lot=32, gamma=0.999):
    if len(liste_observations) < taille_mini_lot:
        return
    observations = liste_observations.mini_lot_observations(taille_mini_lot)
    mini_lot = Observation(*zip(*observations))

    masque_non_final = torch.tensor(
        tuple(s is not None for s in mini_lot.etat_suivant),
        dtype=torch.bool,
    )
    mini_lot_etats = torch.cat(mini_lot.etat)
    mini_lot_actions = torch.cat(mini_lot.action)
    mini_lot_recompenses = torch.cat(mini_lot.recompense)

    mini_lot_Q = rnaq(mini_lot_etats).gather(1, mini_lot_actions)

    mini_lot_Q_suivant = torch.zeros(taille_mini_lot)
    if masque_non_final.any():
        non_final_etat_suivants = torch.cat(
            [s for s in mini_lot.etat_suivant if s is not None]
        )
        mini_lot_Q_suivant[masque_non_final] = (
            rnaq_cible(non_final_etat_suivants).max(1)[0].detach()
        )
    mini_lot_Q_cibles = (mini_lot_Q_suivant * gamma) + mini_lot_recompenses

    cout = F.mse_loss(mini_lot_Q, mini_lot_Q_cibles.unsqueeze(1))
    optimiseur.zero_grad()
    cout.backward()
    for parametre in rnaq.parameters():
        if parametre.grad is not None:
            parametre.grad.data.clamp_(-1, 1)
    optimiseur.step()


def optimiser_DQN(
    env,
    nombre_episodes=40,
    gamma=0.999,
    epsilon_max=1.0,
    epsilon_min=0.05,
    epsilon_taux_decroissance=0.05,
    frequence_maj_rnaq_cible=5,
):
    longueur_episode = np.zeros(nombre_episodes)
    for i_episode in range(nombre_episodes):
        env.reset(seed=i_episode)
        ecran_precedent = chercher_ecran()
        ecran_actuel = chercher_ecran()
        etat = ecran_actuel - ecran_precedent
        epsilon = epsilon_min + (epsilon_max - epsilon_min) * np.exp(
            -epsilon_taux_decroissance * i_episode
        )

        for t in count():
            action = choisir_action(etat, epsilon)
            _, recompense, termine, tronque, _ = env.step(action.item())
            fin_episode = termine or tronque
            recompense_t = torch.tensor([recompense], dtype=torch.float32)
            ecran_precedent = ecran_actuel
            ecran_actuel = chercher_ecran()
            etat_suivant = (
                ecran_actuel - ecran_precedent if not fin_episode else None
            )
            liste_observations.ajouter_historique(
                etat, action, etat_suivant, recompense_t
            )
            etat = etat_suivant
            optimisation_RNAQ(gamma=gamma)
            if fin_episode:
                longueur_episode[i_episode] = t + 1
                break

        if i_episode % frequence_maj_rnaq_cible == 0:
            rnaq_cible.load_state_dict(rnaq.state_dict())

        if (i_episode + 1) % 5 == 0 or i_episode == 0:
            longueur = int(longueur_episode[i_episode])
            print(
                "Episode {}/{}. Longueur:{}".format(
                    i_episode + 1, nombre_episodes, longueur
                )
            )
            sys.stdout.flush()

    return longueur_episode


longueur_episode = optimiser_DQN(
    env,
    nombre_episodes=40,
    gamma=0.999,
    epsilon_max=1.0,
    epsilon_min=0.05,
    epsilon_taux_decroissance=0.05,
    frequence_maj_rnaq_cible=5,
)

env.close()
print("Longueurs (moyenne):", round(float(longueur_episode.mean()), 2))
afficher_longueur_episode(longueur_episode, fenetre=10)